# Nagelhus et al. (2023), single cells: object tuning, fields and whole-trial changes

**Paper.** Nagelhus A, Andersson SO, Gonzalo Cogno S, Moser EI, Moser M-B (2023). Object-centered
population coding in CA1 of the hippocampus. *Neuron* 111, 2091-2104.
[doi:10.1016/j.neuron.2023.04.008](https://doi.org/10.1016/j.neuron.2023.04.008). The claims file
was written from the version of record (open access), with its STAR Methods and Supplemental
information, and from the paper's code on Zenodo (doi:10.5281/zenodo.7797252).

**Dataset.** EBRAINS [10.25493/Q0FG-1X8](https://doi.org/10.25493/Q0FG-1X8), "Extracellular unit
recordings from CA1/CA3 pyramidal cells in rats exploring an open arena with object landmarks",
version v1 (Knowledge Graph id 0162668e-bef0-4754-8562-40440db5bc0c, released 2023-05-05). The paper's
data statement cites this DOI, which is the DOI of version v1 itself, and the Knowledge Graph lists no
other version, so v1 is pinned. Files are streamed from the EBRAINS data proxy into memory.

This is the first of six notebooks. It holds the data check, and the single-cell claims that use
whole-trial rate maps. `02_single_cell_changes.ipynb` holds the within/between-trial classification
and object traces, `03_population_decoding.ipynb` the decoders, `04_distance_tuning.ipynb` the
single-cell decoders, `05_subsets.ipynb` the decoders on subsets of cells, `06_words.ipynb` the
words and counts.

| Claim | Notebook | Tested or blocked |
|---|---|---|
| cell-counts | 01 | tested |
| fig1c-object-tuned-fraction | 01 | tested |
| fig1d-field-distances | 01 | tested |
| fig1e-object-cells-and-object-vector-cells | 01 | tested |
| fig1f-orientation-coverage | 01 | tested |
| fig1g-no-experience-effect | - | blocked: data missing (no cell identities across days, no day of exposure) |
| fig1h-cross-day-stability | - | blocked: data missing (as fig1g) |
| figs2e-within-trial-stability | 01 | tested |
| figs2c-no-proximodistal-effect | - | blocked: data missing (no implant sites) |
| figs3a-ca3-object-tuned | 01 | tested |
| fig2d-spatial-change-no-o | 01 | tested |
| fig2d-rate-overlap-no-o-unchanged | 01 | tested |
| fig2-fraction-changed-no-o | 02 | tested |
| fig2d-o-om-change | 02 | tested |
| fig2d-om-no-prime-change | 01 | tested |
| fig2d-no-no-prime-change | 01 | tested |
| figs3f-ca3-changes | 01 | tested |
| figs4-object-traces | 02 | tested |
| figs2d-traces-no-interference | 02 | tested |
| ovs-method-extra-cells | 01 | tested |
| fig3b to figs5jk (20 decoder claims) | 03 | tested, except figs5d (decided on its selection there) |
| fig6ab to figs8eh, fig7 (6 claims) | 04 | tested |
| fig3d, fig6c, fig6d, figs8 (4 subset claims) | 05 | tested |
| fig5b to figs7e (7 word claims) | 06 | tested |

Every rate map is built as the claims file's opening comment now says ("Smoothing and unvisited
bins", answered on 2026-10-21 from the paper's calculateRateMap.m): the rate per bin (spikes over
time, the time per position sample clipped at 0.1 s), unvisited bins 0, then smoothed; blanked maps
have the unvisited bins set to NaN after smoothing. Tables still print this as `rm` b, the name the
questions file gave it. The other readings run side by side are the claims file's own and are listed
in the docstring of `objcode.py`: `speed` (decoders), `field` (rate criterion of a field), template
width `w` (a, the code's SDs; b, the Methods' variances) and the readings each criterion names.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import objcode as oc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4g}".format)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

started = time.time()
files = oc.list_files()
data, meta = {}, {}
for key, name in list(oc.SINGLE_FILES.items()) + [(p, p) for p in oc.POP_FILES]:
    data[key], meta[key] = oc.load(name)
print(f"dataset {oc.DATASET_ID} version {oc.DATASET_VERSION} (doi {oc.DATASET_DOI}): "
      f"read {len(data)} files in {time.time() - started:.0f} s")
CA1, CA3, CON1, CON3 = data["CA1"], data["CA3"], data["CON_CA1"], data["CON_CA3"]
POP = [s for p in oc.POP_FILES for s in data[p]]

## Data check

In [ ]:
print("Files in the dataset (data proxy listing):")
print(pd.DataFrame(files)[["name", "bytes", "hash", "last_modified"]].to_string(index=False))
rows = []
for key, sess in data.items():
    rows.append({"file": key, "sessions": len(sess), "rats": len({s.rat for s in sess}),
                 "rat ids": " ".join(sorted({s.rat for s in sess})),
                 "techniques": " ".join(sorted({s.technique for s in sess})),
                 "units": sum(s.n_units for s in sess),
                 "trial sets": " | ".join(sorted({"-".join(t.name for t in s.trials) for s in sess})),
                 "unit strings": " ".join(meta[key]["units_strings"])})
print(pd.DataFrame(rows).to_string(index=False))
print("\nFile descriptions:")
for key in data:
    print(f"  {key}: {meta[key]['description']}")
print("\nField names at each level of every file's struct (all sessions, trials and units):")
for key in data:
    print(f"  {key}:")
    for level, names in meta[key]["fields"].items():
        print(f"    {level}: {', '.join(names) if names else '-'}")


The descriptions and counts are checked against the claims file's description of the data: 1,189
CA1 cells from 9 rats (the file says the same), 219 CA1 control cells from 3 rats (same), 59 CA3
control cells from 2 rats (same). The CA3 file holds 341 cells from 4 rats, two of them
Neuropixels sessions; the paper's Supplementary Figure 3 counts 313 CA3 cells. The population files
hold 16 sessions with the No Object and Object trials only (two tetrode rats with 7 sessions each,
and the two Neuropixels sessions), which is the legends' count of 16 decoding sessions (14
tetrode); the Methods' text says 15 tetrode sessions. The stored unit of the tracking (a MATLAB
`string` object, decoded from the file's workspace) is "centimeters" in every file.

In [ ]:
rows = []
for key, sess in data.items():
    for s in sess:
        for t in s.trials:
            rows.append({"file": key, "session": s.index, "rat": s.rat, "tech": s.technique, "trial": t.name,
                         "units": t.n_units, "samples": t.n_samples_raw, "nan_samples": t.n_nan,
                         "dt_ms": 1000 * t.dt, "duration_s": t.duration, "x_min": t.x.min(), "x_max": t.x.max(),
                         "y_min": t.y.min(), "y_max": t.y.max(), "speed_med": np.median(t.speed),
                         "frac_above_2.5": np.mean(t.speed > oc.SPEED_MIN),
                         "obj_x": t.obj[0] if t.obj else np.nan, "obj_y": t.obj[1] if t.obj else np.nan,
                         "unsorted_units": t.n_unsorted,
                         "spikes_no_position": int(sum((si < 0).sum() for si in t.spike_si)),
                         "spikes": int(sum(len(x) for x in t.spikes)),
                         "start": t.t0, "end": t.t1})
trials = pd.DataFrame(rows)
print("Per file and trial type: trials, sampling interval (ms), duration (s), position range (cm), speed:")
summary = trials.groupby(["file", "trial"]).agg(
    n=("units", "size"), dt_ms=("dt_ms", "median"), dur_min=("duration_s", "min"), dur_med=("duration_s", "median"),
    dur_max=("duration_s", "max"), x_min=("x_min", "min"), x_max=("x_max", "max"), y_min=("y_min", "min"),
    y_max=("y_max", "max"), speed_med=("speed_med", "median"), above_2_5=("frac_above_2.5", "median"),
    nan=("nan_samples", "sum"), samples=("samples", "sum"), unsorted=("unsorted_units", "sum"),
    no_pos=("spikes_no_position", "sum"), spikes=("spikes", "sum"))
print(summary.to_string())
print("\nSampling intervals: tetrode", sorted(trials[trials.tech == "tetrodes"].dt_ms.round(2).unique()),
      "ms; Neuropixels", sorted(trials[trials.tech == "neuropixels"].dt_ms.round(2).unique()), "ms")
print("Trial durations (s), all files: min", trials.duration_s.min().round(0), "median",
      trials.duration_s.median().round(0), "max", trials.duration_s.max().round(0),
      "(Methods: typically 20 to 30 min, analyses assume 1800 s)")
print("Units with spike times out of order in some trial:", trials.groupby("file").unsorted_units.sum().to_dict())

Positions lie within 0 to 150 cm in every file, consistent with the 150 cm arena and the stated
unit. Tetrode tracking is sampled every 40 ms (25 Hz, as the Methods say), Neuropixels tracking
every 8.3 ms (120 Hz). In the Neuropixels sessions 8 to 29% of tracking samples have no position
(NaN). They are dropped with their time. A spike is counted at the last position sample at or before
it (histc, as calculateRateMap.m does), and gets no position when that sample has none. These are the
`no_pos` spikes above, nearly all in the Neuropixels sessions, where they are 24 to 29% of spikes in
27012 and 9% in 27207. Three units of the CA1 file have spike times out of order within the Object
Moved trial; they are sorted on loading. Trials last 564 to 2334 s (median 1232 s), shorter than
the 1800 s the Methods assume for their sample counts.

In [ ]:
# Trial order and gaps between trials (Methods: 1-3 min on a pedestal between trials)
gaps = []
for key, sess in data.items():
    for s in sess:
        for a, b in zip(s.trials[:-1], s.trials[1:]):
            gaps.append({"file": key, "session": s.index, "rat": s.rat, "pair": f"{a.name}-{b.name}",
                         "gap_s": b.t0 - a.t1, "ordered": b.t0 > a.t1})
gaps = pd.DataFrame(gaps)
print("Every trial starts after the one before it:", bool(gaps.ordered.all()))
print("Gap between trials (s), by file:")
print(gaps.groupby("file").gap_s.describe().to_string())
long = gaps[gaps.gap_s > 600]
print(f"\nGaps longer than 10 min: {len(long)} of {len(gaps)}")
print(long.to_string(index=False))

Trials are in the protocol's order in every session. The median gap between trials is 81 to 113 s,
as the Methods' 1 to 3 min. But 26 of 273 gaps in the single-cell files are longer than 10 min,
up to 14,737 s (4.1 h), mostly before the Object Moved trial in CA3 and before the Object Moved or
No Object' trial in CA1. The paper does not mention such breaks. No analysis here depends on them, as
each trial is treated on its own.

In [ ]:
# Object positions: Object near the middle; Object Moved 35 to 55 cm from a corner (Methods)
rows = []
for key in ("CA1", "CA3"):
    for s in data[key]:
        o, m = s.trial("O").obj, s.trial("OM").obj
        corner = min(np.hypot(m[0] - cx, m[1] - cy) for cx in (0, 150) for cy in (0, 150))
        corner_o = min(np.hypot(o[0] - cx, o[1] - cy) for cx in (0, 150) for cy in (0, 150))
        rows.append({"file": key, "session": s.index, "rat": s.rat, "O_from_centre": np.hypot(o[0] - 75, o[1] - 75),
                     "OM_from_corner": corner, "O_from_corner": corner_o, "OM_from_centre": np.hypot(m[0] - 75, m[1] - 75)})
objs = pd.DataFrame(rows)
print(objs.groupby("file")[["O_from_centre", "OM_from_corner"]].describe().T.to_string())
print("\nSessions whose Object trial is farther than 30 cm from the centre:")
print(objs[objs.O_from_centre > 30].to_string(index=False))
print("\nOM distance from the nearest corner outside 35-55 cm:", int(((objs.OM_from_corner < 35) | (objs.OM_from_corner > 55)).sum()),
      "of", len(objs), "; range", objs.OM_from_corner.min().round(1), "to", objs.OM_from_corner.max().round(1))

The Object trial's object is within 30 cm of the arena centre in 58 of 60 CA1 sessions. It is
40.4 cm away in CA1 session 0 and 30.2 cm in session 3 (both rat 25224), and more than 30 cm away in
3 of 13 CA3 sessions. In CA3 session 3 (rat 25987) the Object trial's object is 61.4 cm from the
centre, near a corner, and the Object Moved trial's is 9.0 cm from the centre: the reverse of the
protocol. Either that session ran the other way round, or the two positions are swapped. The Object
Moved position is outside the Methods' 35 to 55 cm from a corner in 12 of 73 sessions (29.4 to 97.4
cm).

In [ ]:
# Cell-inclusion rules (Methods): no more than 1% of ISIs under 2 ms; mean rate 0.05-10 Hz over the session
rows = []
for key, sess in data.items():
    for s in sess:
        span = s.trials[-1].t1 - s.trials[0].t0
        dur = sum(t.duration for t in s.trials)
        for u in range(s.n_units):
            sp = np.sort(np.concatenate([t.spikes[u] for t in s.trials]))
            isi = np.diff(sp)
            rows.append({"file": key, "isi_frac": np.mean(isi < 0.002) if len(isi) else 0,
                         "rate_trials": len(sp) / dur, "rate_span": len(sp) / span})
inc = pd.DataFrame(rows)
inc["isi_ok"] = inc.isi_frac <= 0.01
inc["rate_ok_trials"] = inc.rate_trials.between(0.05, 10)
inc["rate_ok_span"] = inc.rate_span.between(0.05, 10)
print(inc.groupby("file")[["isi_ok", "rate_ok_trials", "rate_ok_span"]].agg(["sum", "size"]).to_string())
print("\nISI fraction under 2 ms, max per file:", inc.groupby("file").isi_frac.max().round(4).to_dict())
print("Mean rate over trials (Hz), range per file:", inc.groupby("file").rate_trials.agg(["min", "max"]).round(3).to_dict("index"))

The inclusion rules hold for most units but not all, computing the rate over the trials (or over
the span from the first to the last trial). 4 of the 1,189 CA1 units break the ISI rule and 7 the
rate rule; the CA1 controls have 1 unit outside the rate rule. The Methods say "across the full
recording session", which may include time between trials that is not deposited. The population
files give lower rates for the same units, as they hold two trials only. All deposited units are
kept, as the files are the paper's selection.

In [ ]:
# Recordings deposited more than once: identical No Object spike trains across files and sessions
links = {}
for p in POP:
    s, m = oc.link_population(p, CA1)
    links[p.key] = (s, m)
rows = [{"population session": p.key, "rat": p.rat, "units": p.n_units,
         "same recording in CA1 file": links[p.key][0].index if links[p.key][0] else None,
         "units there": links[p.key][0].n_units if links[p.key][0] else 0,
         "identical trains": len(links[p.key][1])} for p in POP]
lk = pd.DataFrame(rows)
print(lk.to_string(index=False))
print("Population units with an identical train in the CA1 file:", lk["identical trains"].sum(), "of", lk.units.sum())
h = {}
for key, sess in data.items():
    for s in sess:
        for u in range(s.n_units):
            h.setdefault(oc.spike_hash(s.trials[0], u), []).append((key, s.index, u))
dups = [v for v in h.values() if len(v) > 1]
within = [v for v in dups if len({x[0] for x in v}) < len(v)]
pairs = pd.Series([" & ".join(sorted({x[0] for x in v})) for v in dups]).value_counts()
print("\nSpike trains shared by more than one record:", len(dups), "; by files:", pairs.to_dict())
print("Shared within one file:", len(within))

Every population session is a recording that is also in the CA1 single-cell file (same tracking).
The two Neuropixels sessions hold the same 620 and 135 units in both files. Of the 425 tetrode
population units, 142 have the same spike trains as a unit of the CA1 file; the other 283 do not
appear in it. The CA1 file holds each cell once ("unique cells"). The 283 are presumably repeat
recordings of cells that the file gives from another day, but no identifier says which. No spike
train is repeated within a file, and no CA1 unit shares a train with CA3. Each recording is counted
once: single-cell claims use the single-cell files, population claims the population files, and
claims needing a population cell's single-cell values use the 897 linked units.

In [ ]:
# Tracking of the population files against the same recordings in the CA1 file
rows = []
for p in POP:
    cs, m = links[p.key]
    for tp in p.trials:
        tc = cs.trial(tp.name)
        same_t = np.array_equal(tp.t, tc.t)
        rows.append({"population session": p.key, "trial": tp.name, "same timestamps": same_t,
                     "x equal": same_t and np.allclose(tp.x, tc.x), "y equal": same_t and np.allclose(tp.y, tc.y),
                     "pop y == pop x (fraction)": np.mean(tp.x == tp.y), "CA1-file y == x (fraction)": np.mean(tc.x == tc.y),
                     "pop corr(x, y)": np.corrcoef(tp.x, tp.y)[0, 1], "CA1-file corr(x, y)": np.corrcoef(tc.x, tc.y)[0, 1]})
trk = pd.DataFrame(rows)
print(trk.to_string(index=False))
allx = [(key, s.index, t.name) for key, sess in data.items() for s in sess for t in s.trials if np.mean(t.x == t.y) > 0.5]
print("\nTrials in any file where y equals x in more than half the samples:", allx)

In the two Neuropixels population files (rats 27012 and 27207) the tracking y equals x in every
sample. The same recordings in the CA1 single-cell file have the same timestamps and the same x,
with a y of their own (correlation with x 0.01 to 0.18). In the population files the y column is a
copy of x. The tetrode population files agree with the CA1 file in x and y. The decoding notebooks
take the two Neuropixels sessions' tracking from the CA1 file (`objcode.decoding_sessions`). This is a
data error in the deposit.

In [ ]:
# Clock check: spikes and tracking. Stability of rate maps between the halves of the No Object trial
# with the spikes moved in time against the positions (circularly within the trial); a shared clock
# gives the highest stability at no shift.
shifts = [-60, -10, -2, 0, 2, 10, 60]
rows = []
for key in ("CA1", "CA3", "CON_CA1", "CON_CA3"):
    for s in data[key]:
        t = s.trials[0]
        tm = (t.t[0] + t.t[-1]) / 2
        vals = []
        for sh in shifts:
            tt = t.subset(range(t.n_units))
            tt.spikes = [oc.shift_spikes(t, u, sh) if sh else t.spikes[u] for u in range(t.n_units)]
            tt.spike_si = [t._sample_index(x) for x in tt.spikes]
            m1, _ = oc.single_maps(tt, "b", mask=oc.sample_mask(tt, True, None, tm))
            m2, _ = oc.single_maps(tt, "b", mask=oc.sample_mask(tt, True, tm, None))
            vals.append(np.nanmedian([oc.pearson_nan(m1[u], m2[u]) for u in range(t.n_units)]))
        rows.append({"file": key, "session": s.index, "units": s.n_units, **{f"{sh:+d}s": v for sh, v in zip(shifts, vals)}})
clock = pd.DataFrame(rows)
cols = [f"{sh:+d}s" for sh in shifts]
clock["best"] = clock[cols].idxmax(axis=1)
print(clock.groupby("file")[cols].median().to_string())
print("\nSessions where no shift gives the highest median stability, by file:")
print(clock.groupby("file").apply(lambda d: f"{(d.best == '+0s').sum()} of {len(d)}").to_string())
print("Sessions where it does not:")
print(clock[clock.best != "+0s"].to_string(index=False))

Moving the spikes by 2 s against the positions lowers the median half-trial stability from 0.64
to 0.76 down to 0.12 to 0.33, and by 10 s or more it is near zero, in every file. Zero shift gives
the highest median in 89 of 91 sessions. The exceptions are CA1 session 26 (3 units, highest at
+10 s) and CA3 control session 1 (9 units, highest at +2 s). Spikes and tracking share a clock to
within about 2 s. This is consistent with the data, and was not established for those two sessions.
Trial membership is given by the file structure, and every spike of a trial lies within its
tracking span (0 spikes outside, by a check of the raw files not run here; the `no_pos` count above
is spikes whose position sample has no position).

## cell-counts

> The object sessions yielded 1,189 unique CA1 principal cells from 9 rats, the control sessions 219
> CA1 principal cells from 3 rats, and up to 620 CA1 cells were recorded simultaneously.

In [ ]:
n_ca1 = sum(s.n_units for s in CA1)
n_con = sum(s.n_units for s in CON1)
largest = max(s.n_units for s in CA1)
ok = inc.isi_ok & inc.rate_ok_trials
print(f"CA1 object sessions: {n_ca1} cells from {len({s.rat for s in CA1})} rats, {len(CA1)} sessions; "
      f"passing both inclusion rules as computed here: {int(ok[inc.file == 'CA1'].sum())}")
print(f"CA1 control sessions: {n_con} cells from {len({s.rat for s in CON1})} rats, {len(CON1)} sessions; "
      f"passing: {int(ok[inc.file == 'CON_CA1'].sum())}")
print(f"Largest number of CA1 cells in one session: {largest} "
      f"(rat {[s.rat for s in CA1 if s.n_units == largest][0]})")
print("CA1 cells per rat:", pd.Series({r: sum(s.n_units for s in CA1 if s.rat == r) for r in sorted({s.rat for s in CA1})}).to_dict())
print("Reading (a), each cell once: the file holds each cell once, 1189 cells.")
print("Reading (b), every session: cell-sessions are not deposited. The population files add", 283,
      "tetrode units with no match in the CA1 file, for 2 of the 9 rats only.")
for name, val, exp in (("unique CA1 cells", n_ca1, 1189), ("control CA1 cells", n_con, 219), ("largest simultaneous", largest, 620)):
    print(f"  {name}: {val} against {exp} +/- {0.1 * exp:.1f}: {'within' if abs(val - exp) <= 0.1 * exp else 'outside'}")

**Result: reproduced.** The CA1 file holds 1,189 cells from 9 rats and the control file 219 cells
from 3 rats. The largest session has 620 cells (rat 27207). These are the claim's three numbers.
They are the deposited selection; 1,178 of the 1,189 and 218 of the 219 pass both inclusion rules as
computed here. Reading (b), counting cell-sessions, cannot be formed, as cell-sessions are not
deposited.

## fig1c-object-tuned-fraction

> 19.9% of CA1 cells (237 of 1,189) were object-tuned by the template matching procedure,
> significantly more than the 5.03% (11 of 219) found by the same procedure in control trials
> without objects.

Template matching as in the claims file's opening comment, under both template width readings (`w`:
a, the code's SDs; b, the Methods' variances). The rate maps are those the criterion now gives
(answered 2026-10-21): the rate per bin, unvisited bins 0, then smoothed, not blanked.
Controls with the object placed at (75, 75) cm in the trial standing for O and at (35, 120) cm in
the trial standing for OM, readings CON2-CON3 and CON1-CON2.

In [ ]:
t0 = time.time()
TUNE = {}
rows = []
for rm in "b":
    for w in "ab":
        ca1 = oc.object_tuning(CA1, rm, oc.WIDTHS[w], key="CA1")
        TUNE["CA1", rm, w] = ca1
        k = sum(c["tuned"] for c in ca1)
        for ctl, trs in (("CON2-CON3", ("CON2", "CON3")), ("CON1-CON2", ("CON1", "CON2"))):
            con = oc.object_tuning(CON1, rm, oc.WIDTHS[w], trials=trs, objs=((75.0, 75.0), (35.0, 120.0)), key="CON_CA1")
            TUNE["CON", rm, w, ctl] = con
            kc = sum(c["tuned"] for c in con)
            p0 = kc / len(con)
            bt = stats.binomtest(k, len(ca1), p0) if p0 > 0 else None
            rows.append({"rm": rm, "w": w, "controls": ctl, "tuned": k, "cells": len(ca1), "fraction": k / len(ca1),
                         "control tuned": kc, "control cells": len(con), "chance": p0,
                         "binomial p": bt.pvalue if bt else np.nan,
                         "fraction within 0.199+/-0.0199": abs(k / len(ca1) - 0.199) <= 0.0199,
                         "chance within 0.0503+/-0.00503": abs(p0 - 0.0503) <= 0.00503})
fig1c = pd.DataFrame(rows)
fig1c["significant"] = fig1c["binomial p"] < 0.05
fig1c["reproduced"] = fig1c["fraction within 0.199+/-0.0199"] & fig1c["chance within 0.0503+/-0.00503"] & fig1c.significant
print(fig1c.to_string(index=False))
for rm in "b":
    d = fig1c[fig1c.rm == rm]
    print(f"rm {rm}: readings (w x controls) meeting the criterion: {int(d.reproduced.sum())} of {len(d)}; "
          f"CA1 fraction within tolerance in {int(d['fraction within 0.199+/-0.0199'].sum())}, chance within in {int(d['chance within 0.0503+/-0.00503'].sum())}")
print(f"\n({time.time() - t0:.0f} s)")

In [ ]:
rows = []
for (_, rm, w), cells in [(k, v) for k, v in TUNE.items() if k[0] == "CA1"]:
    df = pd.DataFrame(cells)
    per = df.groupby("rat").tuned.agg(["sum", "size"])
    rows.append({"rm": rm, "w": w, **{r: f"{int(a)}/{int(b)}" for r, (a, b) in per.iterrows()}})
print("Object-tuned cells per rat (tuned/cells):")
print(pd.DataFrame(rows).to_string(index=False))
df = pd.DataFrame(TUNE["CA1", "b", "b"])
print("\nSignificant scores per tuned cell (rm b, w b): one only", int((df[df.tuned].n_sig == 1).sum()), "of", int(df.tuned.sum()),
      f"({(df[df.tuned].n_sig == 1).mean():.3f}; paper 11.4%)")
print("\nControl counts against the paper's chance level (11 of 219, 0.0503), two-sided binomial test:")
for (_, rm, w, ctl), cells in [(k, v) for k, v in TUNE.items() if k[0] == "CON"]:
    kc = sum(c["tuned"] for c in cells)
    print(f"  w {w}, {ctl}: {kc} of {len(cells)}, p = {stats.binomtest(kc, len(cells), 0.0503).pvalue:.3g}")

**Result: partially reproduced.** With the rate maps the criterion now gives, no reading meets the
whole criterion. With the Methods' variances (`w` b), 222 of 1,189 CA1 cells (18.7%) are
object-tuned, within 0.199 ± 0.0199, and far above the controls (binomial p = 6e-79 and 2e-37). But
the control level is 9 of 219 (4.11%, CON2-CON3) or 16 of 219 (7.31%, CON1-CON2), outside
0.0503 ± 0.00503. The 9 of 219 does not differ from the paper's 11 of 219 by a binomial test
(p = 0.64), so the chance-level procedure agrees with the paper's within sampling error, and the
miss is of the tolerance. With the code's SDs (`w` a), 331 cells (27.8%) and 26 or 37 controls
(11.9 or 16.9%) are tuned, both outside. The tuned fraction differs between rats (2.3 to 30.3%
under `w` b, printed above). 20 of 222 tuned cells (9.0%) have one significant score, against the
paper's 11.4%.

## fig1d-field-distances

> Firing fields of object-tuned cells lay at a wide range of distances from the object (4.8 to 80.7
> cm, mean 37.7 cm, SD 16.0 cm) and covered all directions around it.

Fields in the 2.5 cm maps of the O (and OM) trial, under both field readings (`field`), which the
criterion allows (answered 2026-10-21: the paper does not say where the 1 Hz rule applies), for the
tuned cells of each template reading. Distance readings: (a) centroid of the nearest field in O;
(b) as (a), averaged over O and OM; (c) peak bin of the nearest field in O.

In [ ]:
FIELDS = {}


def cell_fields(sessions, rm, field):
    out = {}
    for si, s in enumerate(sessions):
        for tn in ("O", "OM"):
            tr = s.trial(tn)
            maps, _ = oc.single_maps(tr, rm)
            for u in range(s.n_units):
                fl = oc.fields(maps[u], 2.5, field)
                out[si, u, tn] = (fl, maps[u], tr.obj)
    return out


for rm in "b":
    for field in "ab":
        FIELDS[rm, field] = cell_fields(CA1, rm, field)


def distances(F, si, u):
    """Distance under readings a, b, c and the angle of the point used, for one cell (NaN without a field)."""
    res = {}
    geo = {}
    for tn in ("O", "OM"):
        fl, m, obj = F[si, u, tn]
        geo[tn] = [oc.field_geometry(idx, m, obj) for idx in fl]
    gO, gM = geo["O"], geo["OM"]
    if gO:
        near = min(gO, key=lambda g: g["d_centroid"])
        res["a"] = (near["d_centroid"], near["angle"])
        nearp = min(gO, key=lambda g: g["d_peak"])
        ang = np.degrees(np.arctan2(nearp["py"] - F[si, u, "O"][2][1], nearp["px"] - F[si, u, "O"][2][0])) % 360
        res["c"] = (nearp["d_peak"], ang)
    dO = min((g["d_centroid"] for g in gO), default=np.nan)
    dM = min((g["d_centroid"] for g in gM), default=np.nan)
    if np.isfinite(dO) or np.isfinite(dM):
        res["b"] = (np.nanmean([dO, dM]), res["a"][1] if "a" in res else min(gM, key=lambda g: g["d_centroid"])["angle"])
    return res


rows = []
DIST = {}
for rm in "b":
    for w in "ab":
        tuned = [(c["session"], c["unit"]) for c in TUNE["CA1", rm, w] if c["tuned"]]
        for field in "ab":
            F = FIELDS[rm, field]
            D = {cu: distances(F, *cu) for cu in tuned}
            DIST[rm, w, field] = D
            for rd in "abc":
                d = np.array([D[cu][rd][0] for cu in tuned if rd in D[cu]])
                ang = np.array([D[cu][rd][1] for cu in tuned if rd in D[cu]])
                sectors = np.bincount((ang // 45).astype(int) % 8, minlength=8)
                rows.append({"rm": rm, "w": w, "field": field, "distance": rd, "tuned": len(tuned), "with field": len(d),
                             "mean": d.mean(), "sd": d.std(ddof=1), "min": d.min(), "max": d.max(),
                             "sectors with a field": int((sectors > 0).sum()), "fewest per sector": int(sectors.min()),
                             "mean ok": abs(d.mean() - 37.7) <= 3.77,
                             "range ok": d.min() >= 4.8 - 0.48 and d.max() <= 80.7 + 8.07,
                             "all sectors": bool((sectors > 0).all())})
fig1d = pd.DataFrame(rows)
fig1d["reproduced"] = fig1d["mean ok"] & fig1d["range ok"] & fig1d["all sectors"]
print(fig1d.to_string(index=False))
print("Readings meeting the criterion:", int(fig1d.reproduced.sum()), "of", len(fig1d))
print(fig1d.groupby(["rm", "field"])[["mean ok", "range ok", "all sectors"]].sum().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
D = DIST["b", "b", "a"]
d = np.array([v["a"][0] for v in D.values() if "a" in v])
a = np.array([v["a"][1] for v in D.values() if "a" in v])
axes[0].hist(d, bins=np.arange(0, 125, 5), color="0.4")
axes[0].set_xlabel("distance of nearest field centroid from object (cm)")
axes[0].set_ylabel("tuned cells")
axes[0].set_title("rm b, w b, field a, distance a")
axes[1] = plt.subplot(122, projection="polar")
axes[1].scatter(np.radians(a), d, s=8, color="k")
plt.tight_layout()
plt.show()
print(f"Plotted: {len(d)} cells, distance {d.min():.1f} to {d.max():.1f} cm, all within the axes")

In [ ]:
# Round 1 additions. (1) The tail of the distance distribution: how many tuned cells have their
# field beyond the paper's largest distance (80.7 cm) and beyond the widened bound (88.8 cm), and
# the range without them. (2) A variant of field detection with bins joined only along edges
# (connectivity 4), not given by the paper, on the same maps, for fig1d and fig1e.
rows = []
for (rm, w, field), D in DIST.items():
    for rd in "abc":
        d = np.array([v[rd][0] for v in D.values() if rd in v])
        rows.append({"w": w, "field": field, "distance": rd, "with field": len(d),
                     "beyond 80.7": int((d > 80.7).sum()), "beyond 88.8": int((d > 88.8).sum()),
                     "below 4.32": int((d < 4.32).sum()), "p95": np.percentile(d, 95), "p99": np.percentile(d, 99),
                     "max without those beyond 88.8": d[d <= 88.8].max()})
print("Tail of the field distances of tuned cells (cm):")
print(pd.DataFrame(rows).to_string(index=False))

rows = []
for field in "ab":
    F = FIELDS["b", field]
    F4 = {k: (oc.fields(m, 2.5, field, connectivity=4), m, obj) for k, (fl, m, obj) in F.items()}
    n8 = sum(len(v[0]) for v in F.values())
    n4 = sum(len(v[0]) for v in F4.values())
    for w in "ab":
        tuned = [(c["session"], c["unit"]) for c in TUNE["CA1", "b", w] if c["tuned"]]
        D4 = {cu: distances(F4, *cu) for cu in tuned}
        for rd in "abc":
            d = np.array([D4[cu][rd][0] if rd in D4[cu] else np.nan for cu in tuned])
            dd = d[np.isfinite(d)]
            ang = np.array([D4[cu][rd][1] for cu in tuned if rd in D4[cu]])
            sectors = np.bincount((ang // 45).astype(int) % 8, minlength=8)
            oc_n, ovc_n, none = int((d < 15).sum()), int((d >= 15).sum()), int(np.isnan(d).sum())
            rows.append({"field": field, "w": w, "distance": rd, "fields (8)": n8, "fields (4)": n4,
                         "mean": dd.mean(), "min": dd.min(), "max": dd.max(), "fewest per sector": int(sectors.min()),
                         "fig1d ok": abs(dd.mean() - 37.7) <= 3.77 and dd.min() >= 4.32 and dd.max() <= 88.77 and bool((sectors > 0).all()),
                         "frac OC": oc_n / 1189, "frac OVC": ovc_n / 1189, "no field": none,
                         "fig1e ok": abs(oc_n / 1189 - 0.041) <= 0.0041 and abs(ovc_n / 1189 - 0.133) <= 0.0133 and abs(none - 30) <= 3})
print("\nVariant: fields with connectivity 4 (fig1d and fig1e):")
print(pd.DataFrame(rows).to_string(index=False))


**Result: partially reproduced.** Fields lie in every 45° sector around the object under every
reading (10 to 24 cells in the sector with fewest), and the mean distance is within 37.7 ± 3.77 cm
under 8 of the 12 readings (29.8 to 40.8 cm over all; for example 34.4 cm, `w` a, field a, distance
b). But the largest distance is 104.2 to 125.3 cm under every reading, beyond the widened bound of
88.8 cm, so no reading meets the whole criterion. The range fails on a few cells: 2 to 19 tuned cells
(1 to 7%) have their field beyond 88.8 cm, and without them the largest distance is 80.4 to 87.9 cm.
The smallest distance is also below 4.32 cm under 6 of the 12 readings (1 to 4 cells). The spread
over cells (SD 16.1 to 26.6 cm) is at or above the paper's 16.0 cm. Both field readings give the same
outcome. Joining field bins only along edges (connectivity 4, a variant) changes the field count by
less than 1% (5,441 to 5,491 and 4,486 to 4,507) and meets the criterion under no reading either.


## fig1e-object-cells-and-object-vector-cells

> About 4.1% of all recorded CA1 cells (49 of 1,189) were object cells, object-tuned with fields under
> 15 cm from the object, and 13.3% (158) were object-vector cells, with fields 15 cm or more away; 30
> object-tuned cells had no detectable field.

In [ ]:
rows = []
for (rm, w, field), D in DIST.items():
    for rd in "abc":
        d = np.array([v[rd][0] if rd in v else np.nan for v in D.values()])
        oc_n, ovc_n, none = int((d < 15).sum()), int((d >= 15).sum()), int(np.isnan(d).sum())
        rows.append({"rm": rm, "w": w, "field": field, "distance": rd, "object cells": oc_n, "frac OC": oc_n / 1189,
                     "OVC": ovc_n, "frac OVC": ovc_n / 1189, "no field": none,
                     "OC ok": abs(oc_n / 1189 - 0.041) <= 0.0041, "OVC ok": abs(ovc_n / 1189 - 0.133) <= 0.0133,
                     "no field ok": abs(none - 30) <= 3})
fig1e = pd.DataFrame(rows)
fig1e["reproduced"] = fig1e["OC ok"] & fig1e["OVC ok"] & fig1e["no field ok"]
print(fig1e.to_string(index=False))
print("Readings meeting the criterion:", int(fig1e.reproduced.sum()), "of", len(fig1e))
fig1e["parts"] = fig1e[["OC ok", "OVC ok", "no field ok"]].sum(axis=1)
print("Most parts within tolerance under one reading, per rm and field:")
print(fig1e.groupby(["rm", "field"]).parts.max().to_string())

**Result: not reproduced.** No reading puts all three values within tolerance, and none puts the
two fractions of the claim within tolerance together. Under `w` a, field a, distance a, 45 object
cells (3.8%) and 30 tuned cells without a field are within tolerance, but 256 object-vector cells
(21.5%) are far above 13.3 ± 1.33%. Under `w` b the object-vector fraction is within tolerance under
three readings (12.1 to 14.4%), with 2.2 to 3.2% object cells, below 4.1 ± 0.41%, and 21 or 52 cells
without a field. Over all readings object cells are 1.9 to 3.8% of CA1 cells, object-vector cells 11.9
to 25.1%, and cells without a field 2 to 72. Both cell types are present under every reading. The
connectivity-4 variant gives 2.0 to 4.0% object cells, 11.8 to 24.8% object-vector cells and 2 to 73
cells without a field, again with no reading within tolerance on all three.


## ovs-method-extra-cells

> The object-vector score method of earlier studies classified 40 cells (3.4%) that the template
> matching did not, mostly false positives on inspection.

The Methods' four criteria (Object-vector score and object-vector cell criteria), with the vector
map of the paper's reference 12 as the criterion now gives it (answer of 2026-10-21): distance bins
of 2 cm, direction bins of 10° from East, the rate per bin smoothed with a Gaussian of SD 1.5 bins,
circular in direction. The rate per bin, its unvisited bins (0 before smoothing, blanked after) and
the speed filter follow the opening comment's rate-map method. Range readings of the criterion:
(r-a) 0 to the largest distance sampled in either trial, the correlation over bins with a value in
both maps; (r-b) 0 to the largest distance sampled in both trials. Shuffles as the Methods' Shuffling
of spike data: each trial's spike train circularly shifted by 20 s to its duration less 20 s, 500
times for the object-vector score (O and OM shifted independently) and 1,000 times per trial for
spatial information; 99th percentiles.

Details the criterion does not fix, run as readings here and asked in the questions file (question
of 2026-10-21): which
fields count for the exclusion of cells whose fields "would fall outside of the arena" in OM (x-a, a
cell is excluded when the centroid of any of its O fields, moved with the object to the OM object
position, lies outside the arena; x-b, when that holds for all of them); the lambda of spatial
information (si-a, si-b, as the fig7 criterion names them); the field rate rule (`field` a, b) and
which field must be 15 cm from the object (distance readings a, b, c of fig1d). The count is of
cells meeting the four criteria and not object-tuned by template matching under each width reading
`w`. Shuffles are run for the cells that have a field in both trials under some field reading, as
the other cells fail criterion 3 under every reading.


In [ ]:
t0 = time.time()
X25, Y25 = oc.bin_centres(2.5)


def outside_after_move(F, si, u, objO, objM):
    """For each O field of the cell: whether its centroid, moved with the object to OM, leaves the arena."""
    fl, m, _ = F[si, u, "O"]
    out = []
    for idx in fl:
        cx = X25[idx].mean() + objM[0] - objO[0]
        cy = Y25[idx].mean() + objM[1] - objO[1]
        out.append(not (0 <= cx <= 150 and 0 <= cy <= 150))
    return out


cand = set()
for field in "ab":
    F = FIELDS["b", field]
    for si, s in enumerate(CA1):
        for u in range(s.n_units):
            if F[si, u, "O"][0] and F[si, u, "OM"][0]:
                cand.add((si, u))
OVS = {}
for si, s in enumerate(CA1):
    tO, tM = s.trial("O"), s.trial("OM")
    mO, mM = oc.max_object_distance(tO, tO.obj), oc.max_object_distance(tM, tM.obj)
    ranges = {"r-a": max(mO, mM), "r-b": min(mO, mM)}
    for u in range(s.n_units):
        if (si, u) not in cand:
            continue
        rng = np.random.default_rng(oc.seed_for("ovs-method-extra-cells", si, u))
        siO = oc.spatial_information_shuffled(tO, u, np.r_[0.0, oc.shift_values(tO, 1000, rng)])
        siM = oc.spatial_information_shuffled(tM, u, np.r_[0.0, oc.shift_values(tM, 1000, rng)])
        shO = np.r_[0.0, oc.shift_values(tO, 500, rng)]
        shM = np.r_[0.0, oc.shift_values(tM, 500, rng)]
        rec = {}
        for k, (a, b) in enumerate(((siO[0], siM[0]), (siO[1], siM[1]))):
            lam = "ab"[k]
            rec[f"si-{lam}"] = bool(a[0] > np.nanpercentile(a[1:], 99) and b[0] > np.nanpercentile(b[1:], 99))
        for rr, dmax in ranges.items():
            r = oc.pearson_rows(oc.vector_maps(tO, u, shO, tO.obj, dmax), oc.vector_maps(tM, u, shM, tM.obj, dmax))
            rec[f"ovs {rr}"] = float(r[0])
            rec[f"thr {rr}"] = float(np.nanpercentile(r[1:], 99))
        OVS[si, u] = rec
print(f"Cells with a field in O and OM under some field reading: {len(cand)} of {n_ca1}; shuffles in {time.time() - t0:.0f} s")
ov = pd.DataFrame.from_dict(OVS, orient="index")
for rr in ("r-a", "r-b"):
    print(f"{rr}: object-vector score median {ov[f'ovs {rr}'].median():.3f} (range {ov[f'ovs {rr}'].min():.2f} to "
          f"{ov[f'ovs {rr}'].max():.2f}), 99th percentile of shuffles median {ov[f'thr {rr}'].median():.3f}; "
          f"above it {int((ov[f'ovs {rr}'] > ov[f'thr {rr}']).sum())} cells")
print("Spatial information above the 99th percentile in both trials:", {k: int(ov[k].sum()) for k in ("si-a", "si-b")})


In [ ]:
rows = []
excl_rows = []
for field in "ab":
    F = FIELDS["b", field]
    for xr in ("x-a", "x-b"):
        excluded = set()
        for si, s in enumerate(CA1):
            objO, objM = s.trial("O").obj, s.trial("OM").obj
            for u in range(s.n_units):
                out = outside_after_move(F, si, u, objO, objM)
                if out and (any(out) if xr == "x-a" else all(out)):
                    excluded.add((si, u))
        excl_rows.append({"field": field, "exclusion": xr, "excluded": len(excluded), "fraction": len(excluded) / n_ca1})
        for w in "ab":
            tuned = {(c["session"], c["unit"]) for c in TUNE["CA1", "b", w] if c["tuned"]}
            D = {cu: distances(F, *cu) for cu in OVS}
            for rr in ("r-a", "r-b"):
                for lam in ("si-a", "si-b"):
                    for rd in "abc":
                        ovc = [cu for cu, rec in OVS.items()
                               if cu not in excluded and rec[lam] and rec[f"ovs {rr}"] > rec[f"thr {rr}"]
                               and F[cu[0], cu[1], "O"][0] and F[cu[0], cu[1], "OM"][0]
                               and rd in D[cu] and D[cu][rd][0] >= 15]
                        extra = [cu for cu in ovc if cu not in tuned]
                        rows.append({"field": field, "exclusion": xr, "w": w, "range": rr, "lambda": lam, "distance": rd,
                                     "OVC by the method": len(ovc), "of them tuned": len(ovc) - len(extra),
                                     "not tuned": len(extra), "fraction": len(extra) / n_ca1,
                                     "within 40+/-4": abs(len(extra) - 40) <= 4})
print("Cells excluded before the criteria (paper: 28.8% of CA1 cells):")
print(pd.DataFrame(excl_rows).to_string(index=False))
ovs = pd.DataFrame(rows)
print(ovs.to_string(index=False))
print("\nReadings with the count within 40 +/- 4:", int(ovs["within 40+/-4"].sum()), "of", len(ovs))
print("Per range reading, readings within tolerance:", ovs.groupby("range")["within 40+/-4"].agg(["sum", "size"]).to_dict("index"))
print("Count not tuned, range over readings:", int(ovs["not tuned"].min()), "to", int(ovs["not tuned"].max()))
print(ovs.groupby(["range", "w"])["not tuned"].agg(["min", "max"]).to_string())


**Result: blocked (methods underspecified).** The count depends on a detail the criterion does
not give: which fields decide the exclusion of cells whose fields would fall outside the arena in
OM. When a cell is excluded if any of its O fields would fall outside (x-a), 59 to 70% of CA1 cells
are excluded and 9 to 15 cells are found by the method and not by template matching, never within
40 ± 4. When it is excluded only if all of them would (x-b), 19 to 21% are excluded and the count is
34 to 53, within tolerance under 27 of the 48 x-b readings, under both range readings of the
criterion (r-a 13, r-b 14 of 48 readings overall). Neither exclusion reading gives the paper's 28.8%
excluded. The object-vector score is above the 99th percentile of its shuffles for 133 (r-a) and 132
(r-b) of the 1,113 cells with fields in both trials, and the two range readings change the count by
at most 1 cell. The manual inspection is not repeated.

## fig1f-orientation-coverage

> The nearer an object-tuned cell's firing field is to the object, the larger the range of
> orientations around the object that its fields cover.

In [ ]:
rows = []
for (rm, w, field), D in DIST.items():
    F = FIELDS[rm, field]
    cov = {cu: oc.orientation_coverage(F[cu[0], cu[1], "O"][0], F[cu[0], cu[1], "O"][2]) for cu in D}
    for rd in "abc":
        pts = [(D[cu][rd][0], cov[cu]) for cu in D if rd in D[cu] and F[cu[0], cu[1], "O"][0]]
        x, y = np.array(pts).T
        r, p = stats.pearsonr(x, y)
        rows.append({"rm": rm, "w": w, "field": field, "distance": rd, "cells": len(x), "r": r, "p": p,
                     "coverage range": f"{y.min():.2f}-{y.max():.2f}", "reproduced": r < 0 and p < 0.05})
fig1f = pd.DataFrame(rows)
print(fig1f.to_string(index=False))
print("Per rm, field and w, any distance reading meeting the criterion:")
print(fig1f.groupby(["rm", "field", "w"]).reproduced.any().to_string())

**Result: reproduced.** Under every `w` and `field` reading, at least one distance reading gives a
negative Pearson correlation between orientation coverage and field distance with p < 0.05. Under
distance reading a, r is -0.52 to -0.73 (p < 1e-12). One combination (`w` b, field b, distance c)
gives r = -0.10, p = 0.19.

## figs2e-within-trial-stability

> Within a single trial, object-tuned cells with larger object-tuning scores were more stable
> (r = 0.45 between within-trial stability and score).

In [ ]:
rows = []
for rm in "b":
    halves = {}
    for si, s in enumerate(CA1):
        tr = s.trial("O")
        tm = (tr.t[0] + tr.t[-1]) / 2
        m1, _ = oc.single_maps(tr, rm, mask=oc.sample_mask(tr, True, None, tm))
        m2, _ = oc.single_maps(tr, rm, mask=oc.sample_mask(tr, True, tm, None))
        for u in range(s.n_units):
            halves[si, u] = oc.pearson_nan(m1[u], m2[u])
    for w in "ab":
        cells = [c for c in TUNE["CA1", rm, w] if c["tuned"]]
        x = np.array([halves[c["session"], c["unit"]] for c in cells])
        y = np.array([c["single"] for c in cells])
        ok = np.isfinite(x) & np.isfinite(y)
        r, p = stats.pearsonr(x[ok], y[ok])
        rows.append({"rm": rm, "w": w, "tuned cells": len(cells), "used": int(ok.sum()), "r": r, "p": p,
                     "stability range": f"{x[ok].min():.2f}-{x[ok].max():.2f}",
                     "r within 0.45+/-0.045": abs(r - 0.45) <= 0.045, "p<0.05": p < 0.05})
figs2e = pd.DataFrame(rows)
print(figs2e.to_string(index=False))

**Result: reproduced.** Within-trial stability of the O trial correlates with the object-tuning
score over the tuned cells: r = 0.41 (`w` a, 330 cells) and 0.42 (`w` b, 221 cells), both within
0.45 ± 0.045, p < 1e-10. One tuned cell under each width has no stability value. With the rate maps
the criterion now gives, the width readings agree; before, they did not.

## figs2c-no-proximodistal-effect (blocked): variant

> The percentage of object-tuned cells did not depend on the proximodistal location of the recording
> site (r = 0.42, p = 0.22).

The claim is blocked: the dataset holds no implant sites. The verifier noted that the Supplementary
Figure 2C legend lists the cell counts in proximodistal order (39, 129, 43, 13, 242, 151, 26, 135,
378, 33), and that every rat's count in the CA1 file matches one of them, except rat 27207, whose
620 cells are the two probes (242 and 378) and are not told apart in its file. That order comes from
the paper, not the dataset, and the claims file does not give it, so it is run here as a variant
only. Rat 27207 is (i) left out (8 implants) or (ii) given the same percentage at both of its
positions (10 implants), and, as a check of the unit order, (iii) its first 242 units are taken as
one probe and the rest as the other, and the reverse.


In [ ]:
LEGEND_ORDER = [39, 129, 43, 13, 242, 151, 26, 135, 378, 33]   # Supplementary Figure 2C legend
rows = []
for w in "ab":
    df = pd.DataFrame(TUNE["CA1", "b", w])
    per = df.groupby("rat").tuned.agg(["sum", "size"])
    pos, unmatched = {}, []
    for rat, (k, n) in per.iterrows():
        hits = [j for j, c in enumerate(LEGEND_ORDER) if c == n]
        if len(hits) == 1:
            pos[rat] = (hits[0], 100 * k / n)
        else:
            unmatched.append((rat, int(n)))
    big = [r for r, n in unmatched if n == 620]
    variants = {"(i) 8 implants, 27207 left out": [v for v in pos.values()]}
    if big:
        sub = df[df.rat == big[0]].sort_values("unit")
        both = 100 * sub.tuned.mean()
        variants["(ii) 10 implants, 27207 same % at both"] = list(pos.values()) + [(4, both), (8, both)]
        t = sub.tuned.to_numpy()
        variants["(iii) first 242 units at position 5"] = list(pos.values()) + [(4, 100 * t[:242].mean()), (8, 100 * t[242:].mean())]
        variants["(iii) first 378 units at position 9"] = list(pos.values()) + [(8, 100 * t[:378].mean()), (4, 100 * t[378:].mean())]
    for name, pts in variants.items():
        x = np.array([q[0] for q in pts], float)
        y = np.array([q[1] for q in pts])
        r, pv = stats.pearsonr(x, y)
        rows.append({"w": w, "variant": name, "implants": len(pts), "r": r, "p": pv,
                     "% tuned in order": " ".join(f"{v:.0f}" for _, v in sorted(pts))})
    print(f"w {w}: rats matched to one legend count: {len(pos)}; unmatched (rat, cells): {unmatched}")
print(pd.DataFrame(rows).to_string(index=False))


**Result: blocked (data missing).** The dataset holds no implant sites, and the claims file does not
give the legend's order, so this variant does not decide the outcome (a question is added). Eight
rats match one count of the Supplementary Figure 2C legend each; rat 27207 (620 cells) matches none,
as its two probes are not told apart. With the Methods' variances (`w` b) every variant gives no
correlation (r = 0.36 to 0.46, p = 0.18 to 0.37), as the paper found (r = 0.42, p = 0.22). With the
code's SDs (`w` a) r is 0.60 to 0.71, with p = 0.035 and 0.022 under two variants and 0.052 and 0.065
under the others. Taking the first 242 or 378 units of rat 27207 as one probe is a guess, as no unit
field gives the probe (data check).


## figs3a-ca3-object-tuned

> Object-related activity was present also in CA3, where 23% of cells (72 of 313) were object-tuned.

In [ ]:
rows = []
for rm in "b":
    for w in "ab":
        ca3 = oc.object_tuning(CA3, rm, oc.WIDTHS[w], key="CA3")
        TUNE["CA3", rm, w] = ca3
        df = pd.DataFrame(ca3)
        k = int(df.tuned.sum())
        tet = df[df.technique == "tetrodes"]
        rows.append({"rm": rm, "w": w, "tuned": k, "cells": len(df), "fraction": k / len(df),
                     "within 0.23+/-0.023": abs(k / len(df) - 0.23) <= 0.023,
                     "tetrode only": f"{int(tet.tuned.sum())}/{len(tet)} = {tet.tuned.mean():.3f}",
                     "per rat": " ".join(f"{r}:{int(g.tuned.sum())}/{len(g)}" for r, g in df.groupby("rat"))})
figs3a = pd.DataFrame(rows)
print(figs3a.to_string(index=False))
print("\nThe CA3 file holds", len(CA3), "sessions and", sum(s.n_units for s in CA3), "cells; the paper counts 313.")
# Round 1: per-session CA3 counts, to compare with the paper's 313 cells
per = []
for si, s in enumerate(CA3):
    row = {"session": s.index, "rat": s.rat, "technique": s.technique, "cells": s.n_units,
           "has NO'": s.trial("NOp") is not None}
    for w in "ab":
        row[f"tuned w {w}"] = int(sum(c["tuned"] for c in TUNE["CA3", "b", w] if c["session"] == si))
    per.append(row)
per = pd.DataFrame(per)
print("\nCA3 cells per session:")
print(per.to_string(index=False))
print("Cells by technique:", per.groupby("technique").cells.sum().to_dict(),
      "; without the largest session:", int(per.cells.sum() - per.cells.max()))
for w in "ab":
    df = pd.DataFrame(TUNE["CA3", "b", w])
    swapped = [si for si, s in enumerate(CA3) if s.index == 3]
    keep = df[~df.session.isin(swapped)]
    print(f"w {w}, without session 3 (object positions reversed): {int(keep.tuned.sum())}/{len(keep)} = {keep.tuned.mean():.3f}")


**Result: not reproduced.** Over the 341 CA3 cells of the file, 19.1% are object-tuned with the
Methods' variances (`w` b) and 32.6% with the code's SDs (`w` a). Both are outside 0.23 ± 0.023. The
file holds 341 cells, not the paper's 313, and its Neuropixels cells (221) are more often tuned than
the tetrode cells (per rat above). Tetrode cells alone give 8.3% (`w` b) and 28.3% (`w` a). The cells
the paper used cannot be identified. The per-session
counts above (2 to 26 tetrode cells, and 57 and 164 Neuropixels cells) do not tell which 28 cells the
paper left out; several sets of sessions sum to 28. Without CA3 session 3, whose object positions are
reversed, the fractions are 18.7% (`w` b) and 31.7% (`w` a), still outside.


## Whole-trial changes between trials (Figure 2D, Supplementary Figure 3F)

Spatial correlation between the 2.5 cm, speed-filtered, blanked maps of two trials, over bins with
a value in both. Rate overlap from the mean rate in each trial: (a) speed-filtered epochs; (b) the
whole trial. Each object-session pair is tested against its control pair with a two-sided Wilcoxon
rank-sum test. The tables print the medians, z, p and the difference in medians.

In [ ]:
def pair_measures(sessions, a, b, rm):
    sc, roa, rob, rats = [], [], [], []
    for s in sessions:
        ta, tb = s.trial(a), s.trial(b)
        if ta is None or tb is None:
            continue
        ma, _ = oc.single_maps(ta, rm)
        mb, _ = oc.single_maps(tb, rm)
        for u in range(s.n_units):
            sc.append(oc.pearson_nan(ma[u], mb[u]))
            roa.append(oc.rate_overlap(oc.mean_rate(ta, u, True), oc.mean_rate(tb, u, True)))
            rob.append(oc.rate_overlap(oc.mean_rate(ta, u, False), oc.mean_rate(tb, u, False)))
            rats.append(s.rat)
    return pd.DataFrame({"sc": sc, "ro_a": roa, "ro_b": rob, "rat": rats})


PAIRS = [("NO", "O", "CON1", "CON2"), ("O", "OM", "CON2", "CON3"), ("OM", "NOp", "CON3", "CON4"), ("NO", "NOp", "CON1", "CON4")]


def compare(obj_sessions, con_sessions):
    rows, store = [], {}
    for rm in "b":
        for a, b, ca, cb in PAIRS:
            X = pair_measures(obj_sessions, a, b, rm)
            Y = pair_measures(con_sessions, ca, cb, rm)
            store[rm, a, b] = (X, Y)
            for meas in ("sc", "ro_a", "ro_b"):
                x, y = X[meas].dropna(), Y[meas].dropna()
                z, p = oc.ranksum(x, y)
                rows.append({"rm": rm, "pair": f"{a}-{b}", "control": f"{ca}-{cb}", "measure": meas,
                             "n obj": len(x), "n con": len(y), "median obj": x.median(), "median con": y.median(),
                             "difference": x.median() - y.median(), "z": z, "p": p})
    return pd.DataFrame(rows), store


T2, S2 = compare(CA1, CON1)
print(T2.to_string(index=False))
print("Cells with a No Object' trial:", int(sum(s.n_units for s in CA1 if s.trial("NOp") is not None)), "of", n_ca1)

In [ ]:
X, Y = S2["b", "NO", "O"]
print("NO-O spatial correlation by rat (median, cells); control CON1-CON2 median", round(Y.sc.median(), 3))
print(X.groupby("rat").sc.agg(["median", "size"]).T.to_string())
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for k, (meas, lab) in enumerate((("sc", "spatial correlation"), ("ro_a", "rate overlap (a)"))):
    for d, c, l in ((X, "k", "NO-O"), (Y, "0.6", "CON1-CON2")):
        v = np.sort(d[meas].dropna())
        ax[k].plot(v, np.arange(1, len(v) + 1) / len(v), color=c, label=l)
    ax[k].set_xlabel(lab)
ax[0].legend()
plt.tight_layout()
plt.show()
print("Spatial correlation range:", X.sc.min().round(2), "to", X.sc.max().round(2), "; rate overlap range:", X.ro_a.min().round(2), "to", X.ro_a.max().round(2))

**Result: reproduced** (fig2d-spatial-change-no-o). NO-O spatial correlations (median 0.45) are
lower than CON1-CON2 correlations (0.80), z = -12.1, p = 6e-34. The median is below the control
median in all 9 rats (0.36 to 0.79).

**Result: blocked (methods underspecified)** (fig2d-rate-overlap-no-o-unchanged). The two
rate-overlap readings give different outcomes. With speed-filtered rates (a), NO-O overlaps are
lower than CON1-CON2 overlaps (medians 0.684 against 0.737, z = -2.48, p = 0.013): the paper's test
finds a difference. With whole-trial rates (b), the difference is not significant (0.718 against
0.748, z = -1.18, p = 0.24).

**Result: reproduced** (fig2d-om-no-prime-change). OM-NO' spatial correlations (median 0.59) are
lower than CON3-CON4 correlations (0.84), z = -12.7, p = 4e-37. The rate-overlap test gives p = 1.00
(a) and 0.75 (b), over the 1,090 cells with a No Object' trial. The "many cells retained firing
fields" part is the object-trace analysis of notebook 02.

**Result: reproduced** (fig2d-no-no-prime-change). NO-NO' spatial correlations (median 0.32) are
lower than CON1-CON4 correlations (0.70), z = -11.3, p = 9e-30.

## figs3f-ca3-changes

> In CA3 too, spatial correlations between trials differed between object and control sessions for
> every trial pair, while rate overlaps did not.

In [ ]:
T3, S3 = compare(CA3, CON3)
print(T3.to_string(index=False))
sc3 = T3[T3.measure == "sc"]
ro3 = T3[T3.measure != "sc"]
for rm in "b":
    print(f"rm {rm}: spatial tests with p < 0.05: {int((sc3[sc3.rm == rm].p < 0.05).sum())} of 4")
for meas in ("ro_a", "ro_b"):
    print(f"{meas}: rate-overlap tests with p < 0.05: {int((ro3[ro3.measure == meas].p < 0.05).sum())} of 4")
print("CA3 tetrode cells only (spatial, rm b):")
tet = [s for s in CA3 if s.technique == "tetrodes"]
Tt, _ = compare(tet, CON3)
print(Tt[(Tt.measure == "sc") & (Tt.rm == "b")][["pair", "n obj", "median obj", "median con", "z", "p"]].to_string(index=False))
print(f"\nTotal time {time.time() - started:.0f} s")

**Result: blocked (methods underspecified).** All four spatial-correlation tests find lower
correlations in object sessions (p at most 1.4e-6, NO-NO'). The rate-overlap readings give different outcomes.
With whole-trial rates (b) no rate-overlap test is significant (p = 0.066 to 0.46), so the claim is
reproduced. With speed-filtered rates (a) the OM-NO' test is significant (p = 0.0081), so it is
partially reproduced. The file holds 341 CA3 cells against the paper's 313. With the 120 tetrode
cells alone, the NO-O and O-OM spatial tests are significant (p = 0.023 and 0.0059) and the other
two are not.

## Summary

| Claim | Outcome |
|---|---|
| cell-counts | reproduced |
| fig1c-object-tuned-fraction | partially reproduced (18.7% tuned, within tolerance; chance level 4.11% or 7.31%, outside) |
| fig1d-field-distances | partially reproduced (all sectors, mean within tolerance, largest distance beyond the bound) |
| fig1e-object-cells-and-object-vector-cells | not reproduced (no reading has the two fractions within tolerance together) |
| ovs-method-extra-cells | blocked, methods underspecified (the exclusion rule decides the count; asked) |
| fig1f-orientation-coverage | reproduced |
| figs2c-no-proximodistal-effect | blocked, data missing (no implant sites; variant from the legend's order shown) |
| figs2e-within-trial-stability | reproduced (r 0.41 and 0.42) |
| figs3a-ca3-object-tuned | not reproduced (19.1 or 32.6% of 341 cells) |
| fig2d-spatial-change-no-o | reproduced |
| fig2d-rate-overlap-no-o-unchanged | blocked, methods underspecified (p = 0.013 or 0.24 by rate-overlap reading) |
| fig2d-om-no-prime-change | reproduced |
| fig2d-no-no-prime-change | reproduced |
| figs3f-ca3-changes | blocked, methods underspecified (rate-overlap readings differ) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: rate maps follow the paper's calculateRateMap.m as the
claims file now gives it, also for the single-cell maps, whose own code is not given. Whether a
field's 1 Hz rule applies to its peak or to every bin (`field`) is not given by the paper, and both
are run. The object-vector method's exclusion rule, field and lambda are not given and are run as
readings (asked). Speed uses a standard choice. The template's form is the claims file's reading,
an isotropic Gaussian at bin centres. Fields are the regions above the threshold with bins joined at
corners; joining them only along edges is shown as a variant.

What was found about the dataset: in the two Neuropixels population files the y coordinate of the
tracking is a copy of x; the CA1 file holds the correct tracking of the same recordings. The counts
match the paper for CA1 and the controls. The CA3 file
holds 341 cells from 4 rats against the paper's 313. The files carry no cell identities across days,
no day of exposure, no implant sites and no head direction. The population files repeat 142 tetrode
cells and all Neuropixels cells of the CA1 file, and add 283 tetrode units that cannot be linked to
it. One CA3 session has its Object and Object Moved positions the reverse of the protocol. Some gaps
between trials last hours. A few units break the inclusion rules as computed here, and three CA1 units
have spike times out of order. Up to 29% of Neuropixels tracking samples are NaN. The unit of the
tracking is stored as a MATLAB `string` object that only MATLAB reads directly.